In [1]:
import pandas as pd
import numpy as np
from rapidfuzz.fuzz import ratio
import re
import src_any_abogados_fuzzymerge_df_er_v1_sec as src
import src_any_abogados_numbersmerge_df_er_v1_sec as src2
import src_any_abogados_emailmerge_df_er_v1_sec as src3
import src_any_abogados_test_df_er_v1_sec as src4

In [2]:
pd.set_option("display.max_rows", None)

In [3]:
def remove_empty_columns(df):

    df_aux = df.copy()
    empty_columns = list(df_aux.columns[df_aux.isna().mean() == 1])   # Almacena en una lista las columnas que están totalmente vacias
    if "Unnamed: 0" in df_aux.columns:
        empty_columns.append("Unnamed: 0")
    df_aux = df_aux.drop(columns=empty_columns)
    return df_aux

In [4]:
#Lectura de tablas
df_gm_o = pd.read_csv("datasets/Dataset_GM_PR.csv")
df_ca_o = pd.read_csv("datasets/datos_abogados.csv") 
df_naics_o = pd.read_csv("datasets/NAICS_Puerto Rico.csv")

df_gm_o = remove_empty_columns(df_gm_o)
df_ca_o = remove_empty_columns(df_ca_o)
df_naics_o = remove_empty_columns(df_naics_o)

df_gm_o["Email"] = df_gm_o["Email"].str.strip().replace(["Not available"], np.nan)

# Agregar columna que indique de dónde viene el registro
df_gm_o["dataset"] = "gm"
df_ca_o["dataset"] = "ca"
df_naics_o["dataset"] = "naics"

In [5]:
df_naics_o[df_naics_o["Name_N"] == "nestor lo presti reyes"]
df_ca_o[df_ca_o["FULL NAME"] == "nestor lo presti reyes"]

,FULL NAME,FNAME,LNAME,colegiacion,rua,correo,tel_residencial,tel_oficina,tel_celular,otro,especialidades,Practice Area,delegacion,State,dataset


In [6]:
#Agregar columna "is Firm" para diferenciar Firma de Abogados
df_gm_o['is Firm'] = df_gm_o['Name'].apply(lambda x: bool(re.search(src.pattern_firma, str(x), flags=re.IGNORECASE)))

In [7]:
len(df_gm_o)

1496

In [8]:
#DataFrame que almacena los registros que son firmas
df_gm_firmas = df_gm_o[df_gm_o["is Firm"]]

#DataFrame filtrado y almacena los registro que son abogados
df_gm_o = df_gm_o[~df_gm_o["is Firm"]]

#Normalización de nombres
df_gm_o["Clean_Name"] = df_gm_o["Name"].apply(lambda x: src.name_normalized(x)).str.lower()
df_ca_o["FULL NAME"] = df_ca_o["FULL NAME"].apply(lambda x: src.name_normalized(x)).str.lower()
df_naics_o["Name_N"] = df_naics_o["Name_N"].apply(lambda x: src.name_normalized(x)).str.lower()

#Reset index
df_gm_o = df_gm_o.reset_index(drop=True)
df_ca_o = df_ca_o.reset_index(drop=True)
df_naics_o = df_naics_o.reset_index(drop=True)

In [9]:
len(df_gm_o)

698

In [10]:
# Columnas importantes: Name, city, state_name, Address, Website,Phone,Google_category,Type,Email,Specialization,Education,ExperiencE
# Columnas a Analizar: zip, state_id, Orig Specialization
# NO Columnas importantes: Google_URL, population,Google_rank,Google_opinions,Google_category,Processed,Name AI

#df_gm_o.isna().mean()

In [11]:
# Columnas importantes: FULL NAME', 'FNAME', 'LNAME
# Columnas a Analizar: 
# NO Columnas importantes:

#df_ca_o.isna().mean()

In [12]:
# Columnas importantes: FULL NAME', 'FNAME', 'LNAME
# Columnas a Analizar: 
# NO Columnas importantes:

#df_naics_o.isna().mean()

In [13]:
df_ca_o = src2.combine_columns_by_priority(df_ca_o,["tel_celular","tel_residencial","tel_oficina","otro"],"telefono")
df_naics_o = src2.combine_columns_by_priority(df_naics_o,["Mobile phone","Direct Phone Number"], "phone")

In [14]:
df_ca_o["telefono"] = df_ca_o["telefono"].apply(lambda phone: src2.normalized_phone(str(phone)) if pd.notnull(phone) else None)
df_naics_o["phone"] = df_naics_o["Mobile phone"].apply(lambda phone: src2.normalized_phone(str(phone)) if pd.notnull(phone) else None)

# Merge por coincidencia exacta de número de contacto

No se obtuvo ninguna coincidencia exacta por número de teléfono entre los 3 datasets.
Se obtuvo 3 coincidencias exactas entre los datasets del colegio de abogados y naics

In [15]:
#Merge entre gm y ca
df_gm_y_df_ca = src2.merge_by_contact_number(df_gm_o, df_ca_o, "Phone", "telefono", "Clean_Name", "FULL NAME")
print(df_gm_y_df_ca[["Clean_Name","FULL NAME","Phone","telefono","score"]].head())


                Clean_Name                FULL NAME    Phone telefono  score
0     luis mercado hidalgo     luis mercado hidalgo  4537437  4537437      1
1   lutgardo acevedo lopez   lutgardo acevedo lopez  4669845  4669845      1
2         edil quiles seda         edil quiles seda  8962727  8962727      1
3  lizandra torres rosario  lizandra torres rosario  6392632  6392632      1
4  vanessa gordils vazquez  vanessa gordils vazquez  8085036  8085036      1


In [16]:
#Merge entre gm y naics
df_gm_y_df_naics = src2.merge_by_contact_number(df_gm_o, df_naics_o, "Phone", "phone", "Clean_Name", "Name_N")
print(df_gm_y_df_naics[["Clean_Name","Name_N","Phone","phone","score"]].head())

               Clean_Name                  Name_N    Phone    phone  score
0       tus documentos pr    roberto velez medina  8507164  8507164      0
1  julian rivera aspinall  julian rivera aspinall  3600094  3600094      1
2         carlos lamoutte         carlos lamoutte  6886036  6886036      1


In [17]:
#Merge entre ca y naics  -->> **  # Salen 3 registros coincidencia exacta **
df_ca_y_df_naics = src2.merge_by_contact_number(df_ca_o, df_naics_o, "telefono", "phone", "FULL NAME", "Name_N")
print(df_ca_y_df_naics[["FULL NAME","Name_N","telefono","phone","score"]].head())               



                 FULL NAME                           Name_N telefono    phone  \
0     esteban mujica cotto              esteban mujicacotto  5184101  5184101   
1  josue castellanos otero  josue emanuel castellanos otero  2995935  2995935   
2  francisco garcia garcia                 francisco garcia  3984898  3984898   

   score  
0      0  
1      0  
2      0  


In [ ]:
#Merge COMPLETO
df_merged_full = src2.merge_by_contact_number(df_gm_y_df_ca, df_naics_o, "Phone", "phone", "FULL NAME", "Name_N")
print(df_merged_full[["Clean_Name","FULL NAME","Phone","telefono","score"]].head())

Empty DataFrame
Columns: [Clean_Name, FULL NAME, Phone, telefono, score]
Index: []


: 

# Merge por coincidencia exacta de Email

In [ ]:
#Merge entre gm y ca    -->> ** 5 coincidencias exactas **
df_gm_y_df_ca = src3.merge_by_email(df_gm_o, df_ca_o, "Email", "correo", "Clean_Name", "FULL NAME")
print(df_gm_y_df_ca[["Clean_Name","FULL NAME","Email","correo","score"]].head())

#Merge entre gm y naics -->> NO COINCIDENCIAS
df_gm_y_df_naics = src3.merge_by_email(df_gm_o, df_naics_o, "Email", "Email Address", "Clean_Name", "Name_N")
print(df_gm_y_df_naics[["Clean_Name","Name_N","Phone","phone","score"]].head())

#Merge entre ca y naics  -->> ** 5 coincidencias exactas
df_ca_y_df_naics = src3.merge_by_email(df_ca_o, df_naics_o, "correo", "Email Address", "FULL NAME", "Name_N")
print(df_ca_y_df_naics[["FULL NAME","Name_N","correo","Email Address","score"]].head())               

#Merge COMPLETO   -->> NO COINCIDENCIAS
df_merged_full = src3.merge_by_email(df_gm_y_df_ca, df_naics_o, "correo", "Email Address", "FULL NAME", "Name_N")
print(df_merged_full[["Clean_Name","FULL NAME","Phone","telefono","score"]].head())

                          Clean_Name                 FULL NAME Email correo  \
0  angel egozcue de quiebras arecibo     maria barrera rosario   NaN    NaN   
1  angel egozcue de quiebras arecibo     martha berrios rivera   NaN    NaN   
2  angel egozcue de quiebras arecibo    yolanda castro borrero   NaN    NaN   
3  angel egozcue de quiebras arecibo  catherine catala lasanta   NaN    NaN   
4  angel egozcue de quiebras arecibo      luis del valle colon   NaN    NaN   

      score  
0  0.370370  
1  0.333333  
2  0.363636  
3  0.315789  
4  0.377358  
                          Clean_Name                          Name_N  \
0  angel egozcue de quiebras arecibo                    larry suckle   
1  angel egozcue de quiebras arecibo  alex giovanni rodriguez negron   
2  angel egozcue de quiebras arecibo                 aura colon sola   
3  angel egozcue de quiebras arecibo                       joe mehra   
4  angel egozcue de quiebras arecibo                     judy arroyo   

        

# Coincidencias exactas por Nombre

In [ ]:
#Merge entre gm y ca    -->> ** 5 coincidencias exactas **
df_gm_y_df_ca = src3.merge_by_email(df_gm_o, df_ca_o, "Clean_Name", "FULL NAME", "Clean_Name", "FULL NAME")
print(df_gm_y_df_ca[["Clean_Name","FULL NAME","score"]].head())

#Merge entre gm y naics -->> ** 5 coincidencias exactas **
df_gm_y_df_naics = src3.merge_by_email(df_gm_o, df_naics_o, "Clean_Name", "Name_N", "Clean_Name", "Name_N")
print(df_gm_y_df_naics[["Clean_Name","Name_N","score"]].head())

#Merge entre ca y naics  -->> ** 5 coincidencias exactas **
df_ca_y_df_naics = src3.merge_by_email(df_ca_o, df_naics_o, "FULL NAME", "Name_N", "FULL NAME", "Name_N")
print(df_ca_y_df_naics[["FULL NAME","Name_N","score"]].head())               

#Merge COMPLETO   -->> NO HAY CONCIDENCIAS  
df_merged_full = src3.merge_by_email(df_gm_y_df_ca, df_naics_o, "Clean_Name", "Name_N", "FULL NAME", "Name_N")
print(df_merged_full[["Clean_Name","FULL NAME","score"]].head())

               Clean_Name               FULL NAME  score
0    luis mercado hidalgo    luis mercado hidalgo    1.0
1  lutgardo acevedo lopez  lutgardo acevedo lopez    1.0
2   jorge hernandez lopez   jorge hernandez lopez    1.0
3   angela oquendo negron   angela oquendo negron    1.0
4   manlio arraiza donate   manlio arraiza donate    1.0
         Clean_Name            Name_N  score
0  felipe sotoortiz  felipe sotoortiz    1.0
1     jaime ruberte     jaime ruberte    1.0
2   gilberto oliver   gilberto oliver    1.0
3      jose benitez      jose benitez    1.0
4   francisco ramos   francisco ramos    1.0
                      FULL NAME                        Name_N  score
0   veronica gonzalez rodriguez   veronica gonzalez rodriguez    1.0
1          edwin rivera cintron          edwin rivera cintron    1.0
2             alba lopez arzola             alba lopez arzola    1.0
3       isabel ruberte figueroa       isabel ruberte figueroa    1.0
4  isamillie melendez caraballo  isamillie 

# Merge por coincidencias difusas de nombre

In [ ]:
df_gm = df_gm_o[["Clean_Name"]]
df_ca = df_ca_o[["FULL NAME"]]
df_naics = df_naics_o[["Name_N"]]

In [ ]:
matches = src.find_fuzzy_matches(df_ca, df_naics,"FULL NAME", "Name_N", 76)

In [ ]:
evaluacion = src4.test(matches)


In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

# Suponiendo que tienes:
y_true = evaluacion["actual"]
y_pred = evaluacion["predicted"]

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# F1-score
f1 = f1_score(y_true, y_pred)

# Mostrar resultados
print(f"TP: {tp}")
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"F1-score: {f1:.2f}")


TP: 185
TN: 799
FP: 24
FN: 155
F1-score: 0.67


In [ ]:
final = evaluacion[evaluacion["predicted"] == 1]
final.reset_index(drop=True, inplace=True)
final.to_csv("datasets/fuzz_merge.csv")

In [ ]:
final.head()

,name1,name2,score,actual,predicted
0,giancarlo rivera cabrera,giancarlo rivera cabrera,100.0,1,1
1,isamillie melendez caraballo,isamillie melendez caraballo,100.0,1,1
2,jose melendez alfonzo,jose melendez alfonzo,100.0,1,1
3,yeidalis collazo salgado,yeidalis collazo salgado,100.0,1,1
4,yelitza rivera aponte,yelitza rivera aponte,100.0,1,1


In [ ]:
#------------------------- PRUEBA CON ARCHIVOS DE CHATGPT --------------------------------#

In [ ]:
step1 = final.merge(df_ca_o, left_on="name1", right_on="FULL NAME", how="inner")
final_step = step1.merge(df_naics_o, left_on="name2", right_on="Name_N", how="inner")
columnas_principales = ["FULL NAME", "Name_N", "score","FNAME","LNAME"]
otras_columnas = [col for col in final_step.columns if col not in columnas_principales]
final_step = final_step[columnas_principales + otras_columnas]
final_step.drop(columns=["name1","name2","actual","predicted"], inplace=True)
final_step = final_step.rename(columns={
    "FULL NAME": "name_colegio_abogados",
    "Name_N": "name_naics"
})
final_step.to_csv("datasets/merge_colegio-abogados_naics.csv")
final_step


,name_colegio_abogados,name_naics,score,FNAME,LNAME,colegiacion,rua,correo,tel_residencial,tel_oficina,...,Company State,Company Zip Code,Company Country,Full Address,Number of Locations,Query Name,Subject 2 New,Body 2 New,dataset_y,phone
0,giancarlo rivera cabrera,giancarlo rivera cabrera,100.000000,Giancarlo,Rivera Cabrera,NaN,NaN,NaN,NaN,NaN,...,Puerto Rico,918,United States,"208 Ave Ponce De Leon Ste 1400, San Juan, Puer...",2.0,naics-541110puertorico,NaN,NaN,naics,None
1,isamillie melendez caraballo,isamillie melendez caraballo,100.000000,Isamillie,Melendez Caraballo,21361.0,22798.0,NaN,NaN,NaN,...,Puerto Rico,908,United States,"PO Box 9134, San Juan, Puerto Rico, 00908, Uni...",5.0,naics-541110puertorico,NaN,NaN,naics,None
2,jose melendez alfonzo,jose melendez alfonzo,100.000000,Jose,Melendez Alfonzo,17772.0,16519.0,NaN,NaN,NaN,...,Puerto Rico,901,United States,"150 Tetuan St, San Juan, Puerto Rico, 00901, U...",14.0,naics-541110puertorico,NaN,NaN,naics,None
3,yeidalis collazo salgado,yeidalis collazo salgado,100.000000,Yeidalis,Collazo Salgado,21458.0,23276.0,NaN,NaN,NaN,...,Puerto Rico,920,United States,"1647 Adams St, San Juan, Puerto Rico, 00920, U...",1.0,naics-541110puertorico,NaN,NaN,naics,None
4,yelitza rivera aponte,yelitza rivera aponte,100.000000,Yelitza,Rivera Aponte,21547.0,23714.0,NaN,NaN,NaN,...,Valencia,46002,Spain,"B · 1 1, Valencia, Valencia, 46002, Spain",4.0,naics-541110puertorico,NaN,NaN,naics,7876050182
5,edwin rivera cintron,edwin rivera cintron,100.000000,Edwin,Rivera Cintron,14332.0,13055.0,erclegal@yahoo.com,NaN,787-716-0618,...,NaN,NaN,United States,United States,NaN,naics-541110puertorico,NaN,NaN,naics,None
6,magda boyles pont,magda boyles pont,100.000000,Magda,Boyles Pont,18835.0,17560.0,NaN,NaN,NaN,...,Puerto Rico,918,United States,"270 Munoz Rivera, San Juan, Puerto Rico, 00918...",9.0,naics-541110puertorico,NaN,NaN,naics,7873546221
7,arianaliz torres cruz,arianaliz torres cruz,100.000000,Arianaliz,Torres Cruz,NaN,NaN,NaN,NaN,NaN,...,Puerto Rico,907,United States,"954 Ave Ponce De Leon Ste 201, San Juan, Puert...",1.0,naics-541110puertorico,NaN,NaN,naics,7879076310
8,jaime sanabria montanez,jaime sanabria montanez,100.000000,Jaime,Sanabria Montañez,15716.0,14260.0,NaN,NaN,NaN,...,Valencia,46002,Spain,"B · 1 1, Valencia, Valencia, 46002, Spain",4.0,naics-541110puertorico,NaN,NaN,naics,7875043004
9,isabel ruberte figueroa,isabel ruberte figueroa,100.000000,Isabel,Ruberte Figueroa,20449.0,21395.0,isabel.ruberte12@gmail.com,NaN,787-815-2680,...,Puerto Rico,908,United States,"PO Box 9134, San Juan, Puerto Rico, 00908, Uni...",5.0,naics-541110puertorico,NaN,NaN,naics,None


In [ ]:
def pruebas(df1, df2, name1, name2, threshold):
    matches = src.find_fuzzy_matches(df1, df2,name1, name2, threshold)
    evaluacion = src4.test(matches)

    # Suponiendo que tienes:
    y_true = evaluacion["actual"]
    y_pred = evaluacion["predicted"]

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # F1-score
    f1 = f1_score(y_true, y_pred)

    # Mostrar resultados
    print(f"TP: {tp}")
    print(f"TN: {tn}")
    print(f"FP: {fp}")
    print(f"FN: {fn}")
    print(f"F1-score: {f1:.2f}")

    return tn, fp, fn, tp, f1

In [ ]:
pruebas(df_gm, df_ca, "Clean_Name", "FULL NAME", 76)


TP: 99
TN: 399
FP: 9
FN: 85
F1-score: 0.68


(np.int64(399), np.int64(9), np.int64(85), np.int64(99), 0.678082191780822)

In [ ]:
pruebas(df_gm, df_naics, "Clean_Name", "Name_N", 76)
pruebas(df_ca, df_naics, "FULL NAME", "Name_N", 76)

TP: 19
TN: 409
FP: 5
FN: 73
F1-score: 0.33
TP: 185
TN: 799
FP: 24
FN: 155
F1-score: 0.67


(np.int64(799), np.int64(24), np.int64(155), np.int64(185), 0.6739526411657559)

In [ ]:
print(len(df_gm))
print(len(df_ca))
print(len(df_naics))

698
2700
1888
